# Jaws Segmentation — Demo

A thin walkthrough of the `jaws_seg` package: load a trained checkpoint, evaluate it on the real test split, and plot a prediction.

For the full training/evaluation/prediction CLI, architecture notes, and real results across all three planes, see the [repo README](../README.md). The original 2022 notebook (all logic inline, before the package rewrite) is preserved at [`../legacy/Jaws Segmentation.ipynb`](../legacy/Jaws%20Segmentation.ipynb).

Run this from the repo root's virtual environment (`pip install -e ".[dev]"`), with the `dataset/` folder present or linked (see README → Getting started).

In [ ]:
import torch

from jaws_seg.config import default_device
from jaws_seg.data.dataset import build_dataset
from jaws_seg.engine import evaluate_detailed
from jaws_seg.models import UNet
from jaws_seg.viz import plot_triptych, predict_mask

PLANE = "coronal"  # one of: axial, coronal, sagittal
CHECKPOINT = f"../checkpoints/{PLANE}/checkpoint_epoch19.pth"
DATA_DIR = "../dataset"
DEVICE = default_device()

model = UNet(n_channels=1, n_classes=3, bilinear=False).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True), strict=True)
model.eval()
print(f"loaded {CHECKPOINT} on {DEVICE}")

## Evaluate on the real test split

Same computation the README's results table numbers come from, via `jaws-evaluate` under the hood.

In [ ]:
test_dataset = build_dataset(PLANE, "test", DATA_DIR, image_height=160, image_width=240)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, num_workers=0)

result = evaluate_detailed(model, test_loader, DEVICE)
result

## Predict on one real sample

In [ ]:
image, true_mask = test_dataset[0]
pred_mask = predict_mask(model, image.unsqueeze(0), DEVICE)

plot_triptych(image, pred_mask, true_mask, title=f"{PLANE} sample 0")